# LaTeX tables

Results, backbones and ablations, written to `figs/*.tex`.

In [ ]:
from pathlib import Path

import utils as u

data_path = Path("../data")
figs_path = Path("figs")

In [ ]:
u.load_data(data_path)
figs_path.mkdir(exist_ok=True)

## Results

LIBERO and LIBERO-plus success of every policy, average and per suite.

In [ ]:
rows = [(lab, role, u.libero_success(run), u.plus_success(spec)) for lab, role, run, spec in u.ranked_policies()]
cols = [[r[2][j] for r in rows] for j in range(5)] + [[r[3][j] for r in rows] for j in range(5)]
best = [max(c) for c in cols]
min_drop = min(r[2][0] - r[3][0] for r in rows)
num = r">{\raggedleft\arraybackslash}X"
lines = [rf"\begin{{tabularx}}{{\linewidth}}{{@{{}}l*{{5}}{{{num}}}@{{\hspace{{3mm}}}}*{{5}}{{{num}}}@{{\hspace{{3mm}}}}{num}@{{}}}}",
         r"\toprule",
         r" & \multicolumn{5}{c}{LIBERO} & \multicolumn{5}{c}{LIBERO-plus} & \\",
         r"\cmidrule(lr){2-6}\cmidrule(lr){7-11}",
         r" & avg & Spat & Obj & Goal & Long & avg & Spat & Obj & Goal & Long & drop \\", r"\midrule"]  # fmt: skip
for i, (lab, role, clean, plus) in enumerate(rows):
    cells = [u.tex_cell(cols[j][i], "{:.1f}" if j in (0, 5) else "{:.0f}", bold=cols[j][i] >= best[j] - 1e-9)
             for j in range(10)]  # fmt: skip
    drop = clean[0] - plus[0]
    cells.append(u.tex_cell(-drop, "{:+.1f}", bold=drop <= min_drop + 1e-9))
    prefix = r"\rowcolor{tint}" if role != "vla" else ""
    lines.append(prefix + u.tex_label(lab) + " & " + " & ".join(cells) + r" \\")
lines += [r"\bottomrule", r"\end{tabularx}"]
(figs_path / "results_table.tex").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

## Backbones

The lookup keyed on each VLA's own VLM before / after LIBERO fine-tuning, next to the VLA. Orange: the lookup on the fine-tuned VLM beats the VLA itself.

In [ ]:
lines = [r"\begin{tabularx}{\linewidth}{@{}l*{6}{>{\raggedleft\arraybackslash}X}@{}}", r"\toprule",
         r" & \multicolumn{3}{c}{LIBERO} & \multicolumn{3}{c}{LIBERO-plus} \\",
         r"\cmidrule(lr){2-4}\cmidrule(lr){5-7}",
         r" & \knnword{} before & \knnword{} after & VLA & \knnword{} before & \knnword{} after & VLA \\", r"\midrule"]  # fmt: skip
for label, before, after, vla in u.backbone_rows():
    cells = []
    for col in (0, 1):
        b, a, v = before[col], after[col], vla[col]
        beats = a >= v
        cells += [u.tex_cell(b, "{:.1f}"), u.tex_cell(a, "{:.1f}", bold=beats, colour="knn" if beats else None),
                  u.tex_cell(v, "{:.1f}", colour="learned")]  # fmt: skip
    lines.append(u.tex_label(label) + " & " + " & ".join(cells) + r" \\")
lines += [r"\bottomrule", r"\end{tabularx}"]
(figs_path / "backbones_table.tex").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

## Ablations

One change to the full method at a time, on the official protocol (2,000 episodes).

In [ ]:
ABLATIONS = {  # config stem in outputs/knn/ablations -> label, in the table's order
    "no_language": "prompt ignored", "encoder_pixels": "raw 16×16 pixels", "demos_10": "10% of demos",
    "demos_25": "25% of demos", "demos_50": "50% of demos", "k1": "k = 1 (replay nearest demo)",
    "encoder_dinov2": "DINOv2-S instead of ResNet-18", "no_wrist": "agent-view camera only",
    "no_agentview": "wrist camera only", "no_mode": "plain weighted mean", "no_ensemble": "no temporal ensembling",
    "no_history": "no history matching", "no_state": "no proprioceptive state", "no_time": "no time-step feature",
    "hard_filter": "hard task filter",
}  # fmt: skip

In [ ]:
abf, full = u.DATA["ablations_full"], u.libero_success("knn:00_base")
labels = [(label.replace("%", r"\%").replace("×", r"$\times$"), stem) for stem, label in ABLATIONS.items()]
labels_k = [(f"$k$ = {int(k[1:])} (13 in the full method)", k) for k in ("k02", "k04", "k08", "k24", "k32")]
lines = [r"\begin{tabularx}{\linewidth}{@{}X*{6}{r}@{}}", r"\toprule",
         r"Change to the full method & avg & Spatial & Object & Goal & Long & $\Delta$ \\", r"\midrule",
         r"none (full method, \texttt{lerobot-eval}) & " + " & ".join(u.tex_cell(v, "{:.1f}") for v in full) + r" & \\"]  # fmt: skip
for group in (labels, labels_k):
    for label, stem in group:
        r = abf[stem]
        cells = [u.tex_cell(v, "{:.1f}") for v in (r["success"], *r["suites"])]
        lines.append(f"{label} & " + " & ".join(cells) + f" & {u.tex_cell(r['success'] - full[0], '{:+.1f}')}" + r" \\")
    if group is labels:
        lines.append(r"\midrule")
lines += [r"\bottomrule", r"\end{tabularx}"]
(figs_path / "ablations_table.tex").write_text("\n".join(lines) + "\n")
print("\n".join(lines))